# JEPCO Smart Meter — XGBoost Forecasting Notebook

**Two models:**
1. **Per-Meter Model** — Forecasts individual meter electricity consumption for the next month, estimates the monthly bill, and assigns a Jordan tariff tier.
2. **Area Aggregate Model** — Forecasts the average aggregate (area-level) electricity consumption across all meters.

**Evaluation:** Both models are benchmarked against a Seasonal Naïve (S-Naïve) baseline using MAE, RMSE, MAPE, and sMAPE across all 6 walk-forward folds.

**Data contract:** Consumes the outputs of `JEPCO_PREPOC.ipynb` — specifically `splits/df_scaled_full.csv` and the per-fold CSVs in `splits/`.

---
**Jordan Subsidised Tariff (JEPCO)**
| Monthly kWh | Rate (fils/kWh) | Tier |
|-------------|-----------------|------|
| 0 – 300     | 50              | T1   |
| 301 – 600   | 100             | T2   |
| > 600       | 200             | T3   |
| ≤ 35 kWh   | Minimum charge 1.75 JOD | — |


## Section 0 — Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
import warnings, os, time
from pathlib import Path

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── Colour palette (matches preprocessing notebook) ──────────────────────────
C_BLUE  = "#378ADD"
C_RED   = "#E24B4A"
C_AMBER = "#EF9F27"
C_GREEN = "#639922"
C_TEAL  = "#1D9E75"
C_GRAY  = "#888780"
C_CORAL = "#D85A30"
C_PURPL = "#7F77DD"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

# ── Jordan Tariff ─────────────────────────────────────────────────────────────
MIN_CHARGE_KWH = 35.0
MIN_CHARGE_JOD = 1.75

RAMADAN_START = pd.Timestamp("2025-03-01")
RAMADAN_END   = pd.Timestamp("2025-03-30")

def calc_bill_jod(kwh: float) -> float:
    """Jordan subsidised stepped tariff. Returns bill in JOD."""
    if kwh <= 0:
        return 0.0
    if kwh <= MIN_CHARGE_KWH:
        return MIN_CHARGE_JOD
    cost, rem = 0.0, float(kwh)
    for cap, rate in [(300, 50), (300, 100), (float("inf"), 200)]:
        used = min(rem, cap)
        cost += used * rate
        rem  -= used
        if rem <= 0:
            break
    return round(cost / 1000, 3)

def tier_label(kwh: float) -> str:
    """Assign Jordan tariff tier string."""
    if kwh <= 300: return "T1 (0–300 kWh)"
    if kwh <= 600: return "T2 (301–600 kWh)"
    return               "T3 (>600 kWh)"

def _fmt_xaxis(ax, n_ticks=8):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=n_ticks))
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

# ── Paths ─────────────────────────────────────────────────────────────────────
SPLITS_DIR = Path(r"splits")

print("✔ Imports complete")
print(f"  XGBoost version : {xgb.__version__}")
print(f"  Splits directory: {SPLITS_DIR.resolve()}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## Section 1 — Load Pre-processed Data

In [ ]:
# =============================================================================
# Load the full scaled dataset and rebuild fold dictionaries
# =============================================================================

print("=" * 65)
print("SECTION 1 — Load Data & Rebuild Walk-Forward Folds")
print("=" * 65)

df_scaled = pd.read_csv(
    SPLITS_DIR / "df_scaled_full.csv",
    parse_dates=["freeze_date"]
)

print(f"  Full dataset     : {len(df_scaled):,} rows  |  {df_scaled['meter_B'].nunique()} meters")
print(f"  Date range       : {df_scaled['freeze_date'].min().date()} → {df_scaled['freeze_date'].max().date()}")
print(f"  Columns          : {list(df_scaled.columns)}")

# ── Rebuild fold splits (mirrors preprocessing Section 12) ───────────────────
FOLDS_DEF = [
    ("2025-04-30", "2025-05-01", "2025-05-31", "2025-06-01", "2025-06-30"),
    ("2025-05-31", "2025-06-01", "2025-06-30", "2025-07-01", "2025-07-31"),
    ("2025-06-30", "2025-07-01", "2025-07-31", "2025-08-01", "2025-08-31"),
    ("2025-07-31", "2025-08-01", "2025-08-31", "2025-09-01", "2025-09-30"),
    ("2025-08-31", "2025-09-01", "2025-09-30", "2025-10-01", "2025-10-31"),
    ("2025-09-30", "2025-10-01", "2025-10-31", "2025-11-01", "2025-11-30"),
]

# Try loading pre-saved CSVs first; fall back to slicing df_scaled
folds_out = []
for i, (tr_end, va_s, va_e, te_s, te_e) in enumerate(FOLDS_DEF):
    tr_end_ts = pd.Timestamp(tr_end) + pd.Timedelta("23h30m")
    va_s_ts, va_e_ts = pd.Timestamp(va_s), pd.Timestamp(va_e) + pd.Timedelta("23h30m")
    te_s_ts, te_e_ts = pd.Timestamp(te_s), pd.Timestamp(te_e) + pd.Timedelta("23h30m")

    train_path = SPLITS_DIR / f"fold{i+1}_train.csv"
    val_path   = SPLITS_DIR / f"fold{i+1}_val.csv"
    test_path  = SPLITS_DIR / f"fold{i+1}_test.csv"

    if train_path.exists():
        df_train = pd.read_csv(train_path, parse_dates=["freeze_date"])
        df_val   = pd.read_csv(val_path,   parse_dates=["freeze_date"])
        df_test  = pd.read_csv(test_path,  parse_dates=["freeze_date"])
    else:
        df_train = df_scaled[df_scaled["freeze_date"] <= tr_end_ts].copy()
        df_val   = df_scaled[(df_scaled["freeze_date"] >= va_s_ts) & (df_scaled["freeze_date"] <= va_e_ts)].copy()
        df_test  = df_scaled[(df_scaled["freeze_date"] >= te_s_ts) & (df_scaled["freeze_date"] <= te_e_ts)].copy()

    folds_out.append({"fold": i+1, "train": df_train, "val": df_val, "test": df_test})

print(f"\n  Folds loaded: {len(folds_out)}")
print(f"  {'Fold':<6} {'Train':>12} {'Val':>10} {'Test':>10}")
print(f"  {'─'*6} {'─'*12} {'─'*10} {'─'*10}")
for fd in folds_out:
    print(f"  {fd['fold']:<6} {len(fd['train']):>12,} {len(fd['val']):>10,} {len(fd['test']):>10,}")

## Section 2 — Feature Definitions & Metric Helpers

In [ ]:
# =============================================================================
# Feature sets — mirrors preprocessing Section 9 exactly
# Target: A+KWH (log1p scaled in preprocessing)
# =============================================================================

TARGET = "A+KWH"

# Features available at forecast time (no leakage)
CYCLICAL_FEATURES = [
    "slot_sin", "slot_cos",
    "dow_sin",  "dow_cos",
    "month_sin","month_cos",
    "year_sin", "year_cos",
]

CALENDAR_FEATURES = [
    "is_weekend", "is_business_hour", "is_ramadan",
]

LAG_FEATURES = [
    "lag_1", "lag_48", "lag_336",
    "roll_mean_48", "roll_std_48", "roll_mean_336",
]

AREA_FEATURES = [
    "area_load_lag_48", "area_load_roll_mean_336",
]

BILLING_FEATURES = [
    "billing_tier",
]

IMPUTE_FLAGS = [
    "is_imputed_linear", "is_imputed_seasonal", "is_imputed_knn",
]

FEATURES_PER_METER = (
    CYCLICAL_FEATURES + CALENDAR_FEATURES + LAG_FEATURES +
    AREA_FEATURES + BILLING_FEATURES + IMPUTE_FLAGS
)

# Filter to columns that actually exist in df_scaled
FEATURES_PER_METER = [f for f in FEATURES_PER_METER if f in df_scaled.columns]

print(f"  Per-meter features  ({len(FEATURES_PER_METER)}): {FEATURES_PER_METER}")

# ── Evaluation metrics ────────────────────────────────────────────────────────
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, label: str = "",
                    y_naive: np.ndarray | None = None) -> dict:
    """Return dict of MAE, RMSE, MAPE, sMAPE, WAPE, and MASE (all in original kWh space)."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    eps  = 1e-6
    mape  = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100
    smape = np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100
    wape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100
    mase = np.nan
    
    if y_naive is not None:
        mae_naive = mean_absolute_error(y_true, y_naive)
        mase = mae / (mae_naive + eps)

    return {
        "Model": label,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE%": mape,
        "sMAPE%": smape,
        "WAPE%": wape,
        "MASE": mase,
    }

def expm1_safe(x):
    """Inverse of log1p; clips to ≥ 0."""
    return np.expm1(np.clip(x, 0, 20))

print("✔ Feature sets and metric helpers defined")

## Section 3 — Seasonal Naïve Baseline (S-Naïve)

In [ ]:
# =============================================================================
# S-Naïve baseline: prediction = value from exactly 1 week ago (lag_336)
# This is the standard benchmark for half-hourly energy data.
# =============================================================================

print("=" * 65)
print("SECTION 3 — Seasonal Naïve Baseline (S-Naïve, lag = 336 slots = 7 days)")
print("=" * 65)

snaive_fold_results = []   # for per-fold summary table

for fd in folds_out:
    df_test = fd["test"].dropna(subset=["lag_336", TARGET])

    y_true = expm1_safe(df_test[TARGET].values)
    y_pred = expm1_safe(df_test["lag_336"].values)

    m = compute_metrics(y_true, y_pred, label="S-Naïve")
    m["Fold"] = fd["fold"]
    m["Test month"] = df_test["freeze_date"].dt.month_name().iloc[0]
    snaive_fold_results.append(m)

snaive_df = pd.DataFrame(snaive_fold_results)
print("\n  S-Naïve per-fold performance (per-meter half-hourly, kWh space):")
print(snaive_df[["Fold","Test month","MAE","RMSE","MAPE%","sMAPE%","WAPE%"]].to_string(index=False))
print(f"\n  Avg MAE  : {snaive_df['MAE'].mean():.4f} kWh")
print(f"  Avg RMSE : {snaive_df['RMSE'].mean():.4f} kWh")
print(f"  Avg MAPE : {snaive_df['MAPE%'].mean():.2f}%")
print(f"  Avg sMAPE: {snaive_df['sMAPE%'].mean():.2f}%")
print(f"  Avg WAPE: {snaive_df['WAPE%'].mean():.2f}%")

# ── Visualise S-Naïve quality for fold 1 (sample meter) ──────────────────────
fd1      = folds_out[0]
test1    = fd1["test"].dropna(subset=["lag_336", TARGET])
mid_demo = test1["meter_B"].value_counts().idxmax()  # most-represented meter

sub_t = test1[test1["meter_B"] == mid_demo].sort_values("freeze_date")
y_t   = expm1_safe(sub_t[TARGET].values)
y_p   = expm1_safe(sub_t["lag_336"].values)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle(f"S-Naïve Baseline — Fold 1 Test Month (Meter {mid_demo})", fontweight="bold")

axes[0].plot(sub_t["freeze_date"], y_t, lw=0.8, color=C_BLUE, label="Actual")
axes[0].plot(sub_t["freeze_date"], y_p, lw=0.8, color=C_RED,  ls="--", alpha=0.8, label="S-Naïve")
axes[0].set_title("Actual vs S-Naïve — Full Test Month"); axes[0].set_ylabel("kWh")
axes[0].legend(); _fmt_xaxis(axes[0])

residuals = y_t - y_p
axes[1].hist(residuals, bins=60, color=C_BLUE, edgecolor="white", lw=0.3)
axes[1].axvline(0, color=C_RED, lw=1.5, ls="--")
axes[1].set_title("S-Naïve Residual Distribution")
axes[1].set_xlabel("Residual (kWh)")

plt.tight_layout(); plt.show()

## Section 4 — Model 1: Per-Meter XGBoost Forecaster

A **single global XGBoost model** trained on all meters simultaneously. The model sees `meter_B` as an integer-encoded categorical feature so it can learn meter-specific behaviour while sharing temporal patterns across the fleet.

In [ ]:
# =============================================================================
# SECTION 4 — Per-Meter XGBoost: feature prep + encoding
# =============================================================================

print("=" * 65)
print("SECTION 4 — Per-Meter XGBoost: Feature Preparation")
print("=" * 65)

# Encode meter_B as integer category (enables tree splits on meter identity)
all_meters = sorted(df_scaled["meter_B"].unique())
meter_enc  = {m: i for i, m in enumerate(all_meters)}

def prep_split(df: pd.DataFrame, features: list) -> tuple:
    """Return (X, y) numpy arrays, dropping rows with any NaN in features or target."""
    df = df.copy()
    df["meter_id"] = df["meter_B"].map(meter_enc).astype("int32")
    cols = features + ["meter_id", TARGET, "freeze_date", "meter_B"]
    df   = df[cols].dropna()
    X    = df[features + ["meter_id"]].values.astype("float32")
    y    = df[TARGET].values.astype("float32")
    return X, y, df

FEATURE_COLS = FEATURES_PER_METER + ["meter_id"]

# Quick sanity-check on fold 1
X_tr, y_tr, _ = prep_split(folds_out[0]["train"], FEATURES_PER_METER)
print(f"  Fold 1 train shape : X={X_tr.shape}  y={y_tr.shape}")
print(f"  Feature columns    : {FEATURE_COLS}")
print(f"  Target             : {TARGET} (log1p-scaled kWh)")

In [ ]:
# =============================================================================
# SECTION 4.1 — XGBoost hyperparameters
# Tuned for half-hourly residential electricity (log1p target, tabular data)
# =============================================================================

XGB_PARAMS_METER = {
    "objective":        "reg:squarederror",
    "tree_method":      "hist",           # fast histogram method
    "n_estimators":     800,
    "learning_rate":    0.05,
    "max_depth":        7,
    "min_child_weight": 5,
    "subsample":        0.80,
    "colsample_bytree": 0.80,
    "reg_alpha":        0.1,              # L1 — feature sparsity
    "reg_lambda":       1.0,              # L2 — weight regularisation
    "gamma":            0.05,             # minimum split gain
    "random_state":     42,
    "n_jobs":           -1,
    "early_stopping_rounds": 30,          # stop if val doesn't improve
}

print("Per-Meter XGBoost Hyperparameters:")
for k, v in XGB_PARAMS_METER.items():
    print(f"  {k:<25}: {v}")

In [ ]:
# =============================================================================
# SECTION 4.2 — Walk-Forward Training & Evaluation (6 folds)
# =============================================================================

print("=" * 65)
print("SECTION 4.2 — Per-Meter XGBoost: 6-Fold Walk-Forward Training")
print("=" * 65)

meter_fold_results  = []   # per-fold metrics
meter_fold_models   = []   # trained models (for inspection)
meter_fold_preds    = []   # (fold_num, df with actual/pred columns)

for fd in folds_out:
    fold_num = fd["fold"]
    t0       = time.time()
    print(f"\n  ── Fold {fold_num} ────────────────────────────────────────────")

    X_tr, y_tr, _ = prep_split(fd["train"], FEATURES_PER_METER)
    X_va, y_va, _ = prep_split(fd["val"],   FEATURES_PER_METER)
    X_te, y_te, df_te_clean = prep_split(fd["test"], FEATURES_PER_METER)

    model = xgb.XGBRegressor(**XGB_PARAMS_METER)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )

    y_pred_log = model.predict(X_te)

    # Inverse log1p to recover kWh space
    y_true_kwh = expm1_safe(y_te)
    y_pred_kwh = expm1_safe(y_pred_log)
    y_naive_kwh = None
    if "lag_336" in df_te_clean.columns:
        y_naive_kwh = expm1_safe(df_te_clean["lag_336"].values)

    m = compute_metrics(
        y_true_kwh,
        y_pred_kwh,
        label="XGBoost-PerMeter",
        y_naive=y_naive_kwh,
    )
    m["Fold"]        = fold_num
    m["Test month"]  = fd["test"]["freeze_date"].dt.month_name().iloc[0]
    m["Best iter"]   = model.best_iteration
    meter_fold_results.append(m)

    # Store predictions for later visualisations
    df_preds = df_te_clean.copy()
    df_preds["y_true_kwh"] = y_true_kwh
    df_preds["y_pred_kwh"] = y_pred_kwh
    meter_fold_preds.append((fold_num, df_preds))
    meter_fold_models.append(model)

    print(f"    Rows used   → Train:{len(X_tr):,}  Val:{len(X_va):,}  Test:{len(X_te):,}")
    print(f"    Best iter   : {model.best_iteration}")
    print(f"    MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"MAPE={m['MAPE%']:.2f}%  sMAPE={m['sMAPE%']:.2f}%  "
          f"WAPE"f"={m['WAPE%']:.2f}%  MASE={m['MASE']:.4f}  "
          f"({time.time()-t0:.1f}s)")

meter_results_df = pd.DataFrame(meter_fold_results)

print("\n" + "=" * 65)
print(f"  Mean sMAPE  : {meter_results_df['sMAPE%'].mean():.2f}%")

print("  Per-Meter XGBoost — Full Walk-Forward Summary")
print(f"  Mean MAPE   : {meter_results_df['MAPE%'].mean():.2f}%")

print("=" * 65)
print(f"  Mean RMSE   : {meter_results_df['RMSE'].mean():.4f}")

print(f"  Mean WAPE   : {meter_results_df['WAPE%'].mean():.2f}%")
print(f"  Mean MASE   : {meter_results_df['MASE'].mean():.4f}")

print(meter_results_df[["Fold","Test month","MAE","RMSE","MAPE%","sMAPE%","WAPE%","MASE","Best iter"]].to_string(index=False))
print(f"\n  Mean MAE    : {meter_results_df['MAE'].mean():.4f}")

In [ ]:
fold1_num, fold1_preds = meter_fold_preds[0]
print(fold1_preds.columns.tolist())


In [ ]:
# =============================================================================
# SECTION 4.3 — Per-Meter Model Visualisations
# =============================================================================

print("=" * 65)
print("SECTION 4.3 — Per-Meter Visualisations")
print("=" * 65)

# ── Fig 1: Learning curves (train vs val RMSE) ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle("Per-Meter XGBoost — Learning Curves per Fold", fontweight="bold")
axes = axes.flatten()

for i, (model, fd) in enumerate(zip(meter_fold_models, folds_out)):
    evals = model.evals_result()
    train_rmse = evals["validation_0"]["rmse"]  # validation_0 = val set
    ax = axes[i]
    ax.plot(train_rmse, lw=1.2, color=C_BLUE, label="Val RMSE")
    ax.axvline(model.best_iteration, color=C_RED, ls="--", lw=1,
               label=f"Best @ iter {model.best_iteration}")
    ax.set_title(f"Fold {fd['fold']} — {fd['test']['freeze_date'].dt.month_name().iloc[0]}")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("RMSE (log scale)")
    ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

# ── Fig 2: Actual vs Predicted for Fold 1 (sample meter) ─────────────────────
fold1_num, fold1_preds = meter_fold_preds[0]
mid_demo = fold1_preds["meter_id"].value_counts().idxmax()
sub = fold1_preds[fold1_preds["meter_id"] == mid_demo].sort_values("freeze_date")

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
fig.suptitle(f"Per-Meter XGBoost — Fold 1 Test Month | Meter {mid_demo}", fontweight="bold")

axes[0].plot(sub["freeze_date"], sub["y_true_kwh"], lw=0.8, color=C_BLUE, label="Actual kWh")
axes[0].plot(sub["freeze_date"], sub["y_pred_kwh"], lw=0.8, color=C_RED, ls="--", alpha=0.85,
             label="XGBoost Predicted")
axes[0].set_title("Actual vs Predicted — Full Test Month")
axes[0].set_ylabel("kWh (per 30-min slot)")
axes[0].legend(); _fmt_xaxis(axes[0])

resid = sub["y_true_kwh"].values - sub["y_pred_kwh"].values
axes[1].fill_between(sub["freeze_date"], resid, alpha=0.4, color=C_AMBER)
axes[1].axhline(0, color=C_GRAY, lw=1)
axes[1].set_title("Residuals (Actual − Predicted)")
axes[1].set_ylabel("kWh")
_fmt_xaxis(axes[1])

plt.tight_layout(); plt.show()

# ── Fig 3: Scatter actual vs predicted (all folds, sample) ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Per-Meter XGBoost — Actual vs Predicted Scatter per Fold", fontweight="bold")
axes = axes.flatten()

for i, (fold_num, df_p) in enumerate(meter_fold_preds):
    # sample 5000 points for clarity
    samp = df_p.sample(min(5000, len(df_p)), random_state=42)
    ax = axes[i]
    ax.scatter(samp["y_true_kwh"], samp["y_pred_kwh"],
               s=2, alpha=0.4, color=C_TEAL)
    lim = max(samp["y_true_kwh"].max(), samp["y_pred_kwh"].max()) * 1.05
    ax.plot([0, lim], [0, lim], color=C_RED, lw=1, ls="--", label="Perfect")
    fd = folds_out[i]
    m  = meter_fold_results[i]
    ax.set_title(f"Fold {fold_num} — MAE={m['MAE']:.3f} kWh  RMSE={m['RMSE']:.3f}")
    ax.set_xlabel("Actual kWh")
    ax.set_ylabel("Predicted kWh")
    ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

# ── Fig 4: Feature importance (last fold model) ───────────────────────────────
best_model   = meter_fold_models[-1]   # last fold — largest training window
fi_scores    = best_model.feature_importances_
fi_names     = FEATURE_COLS
fi_df        = pd.DataFrame({"feature": fi_names, "importance": fi_scores})
fi_df        = fi_df.sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [C_RED if v > fi_df["importance"].quantile(0.75) else
          C_AMBER if v > fi_df["importance"].median() else C_BLUE
          for v in fi_df["importance"]]
ax.barh(fi_df["feature"], fi_df["importance"], color=colors, edgecolor="white", lw=0.3)
ax.set_title("Per-Meter XGBoost — Feature Importance (Fold 6, Gain)",
             fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout(); plt.show()

## Section 5 — Next-Month Forecast + Bill Estimation (Per-Meter)

In [ ]:
# =============================================================================
# SECTION 5 — Monthly forecast: aggregate half-hourly predictions → monthly
# kWh → bill in JOD → tier categorisation
#
# Strategy: use the best (last) fold model on the last available test month.
# For a *true* next-month forecast you would roll the model forward on new data;
# here we demonstrate on the held-out last test fold.
# =============================================================================

print("=" * 65)
print("SECTION 5 — Monthly Bill Estimation & Tier Classification")
print("=" * 65)

last_fold_num, last_preds = meter_fold_preds[-1]   # fold 6 (most recent)

# Aggregate half-hourly predictions to monthly kWh per meter
monthly_pred = (
    last_preds.groupby("meter_B")
    .agg(
        actual_monthly_kwh  = ("y_true_kwh", "sum"),
        forecast_monthly_kwh= ("y_pred_kwh",  "sum"),
    )
    .reset_index()
)

# Apply Jordan tariff to both actual and predicted
monthly_pred["actual_bill_jod"]   = monthly_pred["actual_monthly_kwh"].apply(calc_bill_jod)
monthly_pred["forecast_bill_jod"] = monthly_pred["forecast_monthly_kwh"].apply(calc_bill_jod)
monthly_pred["actual_tier"]       = monthly_pred["actual_monthly_kwh"].apply(tier_label)
monthly_pred["forecast_tier"]     = monthly_pred["forecast_monthly_kwh"].apply(tier_label)
monthly_pred["tier_correct"]      = (monthly_pred["actual_tier"] == monthly_pred["forecast_tier"]).astype(int)
monthly_pred["bill_error_jod"]    = (monthly_pred["forecast_bill_jod"] - monthly_pred["actual_bill_jod"]).abs()

tier_accuracy  = monthly_pred["tier_correct"].mean() * 100
mean_bill_err  = monthly_pred["bill_error_jod"].mean()
median_bill_err= monthly_pred["bill_error_jod"].median()

print(f"  Test month (Fold 6): {folds_out[-1]['test']['freeze_date'].dt.month_name().iloc[0]} 2025")
print(f"  Meters evaluated   : {len(monthly_pred)}")
print(f"  Tier classification accuracy : {tier_accuracy:.1f}%")
print(f"  Mean absolute bill error     : {mean_bill_err:.3f} JOD")
print(f"  Median absolute bill error   : {median_bill_err:.3f} JOD")
print(f"\n  Sample forecast (first 10 meters):")
print(monthly_pred.head(20)[["meter_B","actual_monthly_kwh","forecast_monthly_kwh",
                              "actual_bill_jod","forecast_bill_jod",
                              "actual_tier","forecast_tier","tier_correct"]].to_string(index=False))

In [ ]:
# =============================================================================
# SECTION 5.1 — Bill estimation visualisations
# =============================================================================

tier_order  = ["T1 (0–300 kWh)", "T2 (301–600 kWh)", "T3 (>600 kWh)"]
tier_colors = [C_GREEN, C_AMBER, C_RED]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Per-Meter Monthly Bill Estimation (Fold 6 Test Month)", fontweight="bold")

# 1. Actual vs Predicted monthly kWh scatter
ax = axes[0, 0]
for tier, col in zip(tier_order, tier_colors):
    sub_t = monthly_pred[monthly_pred["actual_tier"] == tier]
    ax.scatter(sub_t["actual_monthly_kwh"], sub_t["forecast_monthly_kwh"],
               s=20, alpha=0.6, color=col, label=tier)
lim = monthly_pred[["actual_monthly_kwh","forecast_monthly_kwh"]].max().max() * 1.05
ax.plot([0, lim], [0, lim], color=C_GRAY, lw=1.2, ls="--")
ax.set_xlabel("Actual monthly kWh")
ax.set_ylabel("Forecast monthly kWh")
ax.set_title("Monthly kWh — Actual vs Forecast (coloured by actual tier)")
ax.legend(fontsize=7)

# 2. Actual vs Predicted bill JOD scatter
ax = axes[0, 1]
ax.scatter(monthly_pred["actual_bill_jod"], monthly_pred["forecast_bill_jod"],
           s=20, alpha=0.5, color=C_BLUE)
lim_b = monthly_pred[["actual_bill_jod","forecast_bill_jod"]].max().max() * 1.05
ax.plot([0, lim_b], [0, lim_b], color=C_RED, lw=1.2, ls="--")
ax.set_xlabel("Actual Bill (JOD)")
ax.set_ylabel("Forecast Bill (JOD)")
ax.set_title(f"Bill Estimation — Actual vs Forecast\nMean error: {mean_bill_err:.3f} JOD")

# 3. Bill error distribution
ax = axes[0, 2]
ax.hist(monthly_pred["bill_error_jod"], bins=40, color=C_TEAL, edgecolor="white", lw=0.3)
ax.axvline(mean_bill_err, color=C_RED, lw=1.5, ls="--",
           label=f"Mean error {mean_bill_err:.3f} JOD")
ax.set_title("Distribution of Absolute Bill Error (JOD)")
ax.set_xlabel("|Forecast Bill − Actual Bill| (JOD)")
ax.legend()

# 4. Tier confusion matrix (counts)
ax = axes[1, 0]
confusion = pd.crosstab(
    monthly_pred["actual_tier"], monthly_pred["forecast_tier"],
    rownames=["Actual"], colnames=["Forecast"]
).reindex(index=tier_order, columns=tier_order, fill_value=0)
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", ax=ax,
            linewidths=0.5, cbar=False)
ax.set_title(f"Tier Confusion Matrix\n(Accuracy: {tier_accuracy:.1f}%)")

# 5. Mean forecast bill by tier
ax = axes[1, 1]
tier_agg = monthly_pred.groupby("actual_tier").agg(
    actual_mean=("actual_bill_jod", "mean"),
    forecast_mean=("forecast_bill_jod", "mean")
).reindex(tier_order)
x = np.arange(len(tier_order))
w = 0.35
ax.bar(x - w/2, tier_agg["actual_mean"],   width=w, color=C_BLUE,  label="Actual",   edgecolor="white")
ax.bar(x + w/2, tier_agg["forecast_mean"], width=w, color=C_AMBER, label="Forecast", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(tier_order, fontsize=8)
ax.set_title("Mean Monthly Bill by Tier (Actual vs Forecast)")
ax.set_ylabel("Mean Bill (JOD)")
ax.legend()

# 6. Tier distribution — actual vs forecast
ax = axes[1, 2]
actual_counts   = monthly_pred["actual_tier"].value_counts().reindex(tier_order, fill_value=0)
forecast_counts = monthly_pred["forecast_tier"].value_counts().reindex(tier_order, fill_value=0)
ax.bar(x - w/2, actual_counts.values,   width=w, color=C_BLUE,  label="Actual",   edgecolor="white")
ax.bar(x + w/2, forecast_counts.values, width=w, color=C_AMBER, label="Forecast", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(tier_order, fontsize=8)
ax.set_title("Tier Distribution — Actual vs Forecast")
ax.set_ylabel("Number of Meters")
ax.legend()

plt.tight_layout(); plt.show()
print("\n  Summary Table — Monthly Forecast per Meter (first 20):")
print(monthly_pred.head(20).to_string(index=False))

## Section 6 — Model 1 vs S-Naïve: Comparison

In [ ]:
# =============================================================================
# SECTION 6 — Per-Meter XGBoost vs S-Naïve: Side-by-Side Comparison
# =============================================================================

print("=" * 65)
print("SECTION 6 — Per-Meter XGBoost vs S-Naïve: Full Comparison")
print("=" * 65)

# Merge results tables on Fold
comp_df = (
    snaive_df[["Fold","Test month","MAE","RMSE","MAPE%","sMAPE%"]]
    .rename(columns={"MAE":"SN_MAE","RMSE":"SN_RMSE","MAPE%":"SN_MAPE","sMAPE%":"SN_sMAPE"})
    .merge(
        meter_results_df[["Fold","MAE","RMSE","MAPE%","sMAPE%"]]
        .rename(columns={"MAE":"XGB_MAE","RMSE":"XGB_RMSE","MAPE%":"XGB_MAPE","sMAPE%":"XGB_sMAPE"}),
        on="Fold"
    )
)
comp_df["RMSE_imp%"] = (comp_df["SN_RMSE"] - comp_df["XGB_RMSE"]) / comp_df["SN_RMSE"] * 100
comp_df["MAE_imp%"]  = (comp_df["SN_MAE"]  - comp_df["XGB_MAE"])  / comp_df["SN_MAE"]  * 100

print(comp_df.to_string(index=False))
print(f"\n  Avg RMSE improvement over S-Naïve: {comp_df['RMSE_imp%'].mean():.1f}%")
print(f"  Avg MAE  improvement over S-Naïve: {comp_df['MAE_imp%'].mean():.1f}%")

# ── Visualisation ─────────────────────────────────────────────────────────────
folds_x  = comp_df["Fold"].values
months_x = comp_df["Test month"].values

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle("Per-Meter XGBoost vs Seasonal Naïve — All Folds", fontweight="bold")

w = 0.35
x = np.arange(len(folds_x))

for ax, (sn_col, xgb_col, ylabel) in zip(
    axes.flatten(),
    [("SN_MAE","XGB_MAE","MAE (kWh)"),
     ("SN_RMSE","XGB_RMSE","RMSE (kWh)"),
     ("SN_MAPE","XGB_MAPE","MAPE (%)"),
     ("SN_sMAPE","XGB_sMAPE","sMAPE (%)")]
):
    ax.bar(x - w/2, comp_df[sn_col],  width=w, color=C_GRAY,  label="S-Naïve",  edgecolor="white")
    ax.bar(x + w/2, comp_df[xgb_col], width=w, color=C_BLUE,  label="XGBoost",  edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels([f"F{f}\n{m[:3]}" for f, m in zip(folds_x, months_x)], fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} — XGBoost vs S-Naïve")
    ax.legend()

plt.tight_layout(); plt.show()

# ── Improvement % bar chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.suptitle("RMSE & MAE % Improvement of XGBoost over S-Naïve per Fold", fontweight="bold")

ax.bar(x - w/2, comp_df["RMSE_imp%"], width=w, color=C_TEAL,  label="RMSE improvement %", edgecolor="white")
ax.bar(x + w/2, comp_df["MAE_imp%"],  width=w, color=C_CORAL, label="MAE improvement %",  edgecolor="white")
ax.axhline(0, color=C_GRAY, lw=1)
ax.set_xticks(x)
ax.set_xticklabels([f"F{f}\n{m[:3]}" for f, m in zip(folds_x, months_x)], fontsize=9)
ax.set_ylabel("% Improvement (positive = better than S-Naïve)")
ax.legend()
plt.tight_layout(); plt.show()

## Section 7 — Model 2: Area Aggregate XGBoost Forecaster

This model predicts the **mean electricity consumption across all meters** at each half-hour timestamp — the area-level signal. It operates on a single aggregated time-series (one row per timestamp).

In [ ]:
# =============================================================================
# SECTION 7 — Area Aggregate Series Construction
# =============================================================================

print("=" * 65)
print("SECTION 7 — Area Aggregate Time-Series Construction")
print("=" * 65)

# Compute area series: mean of A+KWH across all meters per timestamp (log1p space)
area_series = (
    df_scaled.groupby("freeze_date")["A+KWH"]
    .mean()
    .reset_index()
    .rename(columns={"A+KWH": "area_kwh_log"})
    .sort_values("freeze_date")
    .reset_index(drop=True)
)

print(f"  Area series length  : {len(area_series):,} timestamps")
print(f"  Date range          : {area_series['freeze_date'].min().date()} → {area_series['freeze_date'].max().date()}")

# ── Feature engineering for the aggregate series ─────────────────────────────
fd = area_series["freeze_date"]
slot = fd.dt.hour * 2 + (fd.dt.minute == 30).astype(int)

area_series["slot_sin"]  = np.sin(2*np.pi*slot/48).astype("float32")
area_series["slot_cos"]  = np.cos(2*np.pi*slot/48).astype("float32")
area_series["dow_sin"]   = np.sin(2*np.pi*fd.dt.dayofweek/7).astype("float32")
area_series["dow_cos"]   = np.cos(2*np.pi*fd.dt.dayofweek/7).astype("float32")
area_series["month_sin"] = np.sin(2*np.pi*(fd.dt.month-1)/12).astype("float32")
area_series["month_cos"] = np.cos(2*np.pi*(fd.dt.month-1)/12).astype("float32")
area_series["year_sin"]  = np.sin(2*np.pi*fd.dt.dayofyear/365.25).astype("float32")
area_series["year_cos"]  = np.cos(2*np.pi*fd.dt.dayofyear/365.25).astype("float32")

area_series["is_weekend"]       = fd.dt.dayofweek.isin([4, 5]).astype("int8")
area_series["is_business_hour"] = fd.dt.hour.between(8, 15).astype("int8")
area_series["is_ramadan"]       = fd.between(RAMADAN_START, RAMADAN_END).astype("int8")
area_series["day_of_year"]      = fd.dt.dayofyear.astype("int16")

# Lag features (strictly causal)
s = area_series["area_kwh_log"]
area_series["lag_1"]         = s.shift(1)
area_series["lag_48"]        = s.shift(48)
area_series["lag_336"]       = s.shift(336)
area_series["roll_mean_48"]  = s.shift(1).rolling(48,  min_periods=12).mean()
area_series["roll_std_48"]   = s.shift(1).rolling(48,  min_periods=12).std()
area_series["roll_mean_336"] = s.shift(1).rolling(336, min_periods=48).mean()

area_series = area_series.dropna().reset_index(drop=True)

AREA_FEATURE_COLS = [
    "slot_sin", "slot_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "year_sin", "year_cos",
    "is_weekend", "is_business_hour", "is_ramadan", "day_of_year",
    "lag_1", "lag_48", "lag_336",
    "roll_mean_48", "roll_std_48", "roll_mean_336",
]

print(f"  After NaN drop      : {len(area_series):,} rows")
print(f"  Area features       : {AREA_FEATURE_COLS}")

# ── Quick visualisation of area series ───────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
fig.suptitle("Area Aggregate Series (Mean A+KWH across all meters)", fontweight="bold")

area_kwh_true = expm1_safe(area_series["area_kwh_log"].values)
axes[0].plot(area_series["freeze_date"], area_kwh_true, lw=0.6, color=C_BLUE)
axes[0].set_title("Area Mean kWh per 30-min slot (original scale)")
axes[0].set_ylabel("Mean kWh")

roll7d = pd.Series(area_kwh_true).rolling(336, center=True).mean()
axes[1].plot(area_series["freeze_date"], area_kwh_true, lw=0.4, color=C_BLUE, alpha=0.5, label="Raw")
axes[1].plot(area_series["freeze_date"], roll7d.values, lw=1.8, color=C_RED, label="7-day rolling mean")
axes[1].axvspan(RAMADAN_START, RAMADAN_END, alpha=0.15, color=C_AMBER, label="Ramadan")
axes[1].set_title("7-day Trend with Ramadan Period")
axes[1].set_ylabel("Mean kWh")
axes[1].legend(); _fmt_xaxis(axes[1])

plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# SECTION 7.1 — S-Naïve baseline for area series (lag_336)
# =============================================================================

print("=" * 65)
print("SECTION 7.1 — Area S-Naïve Baseline")
print("=" * 65)

AREA_FOLDS_DEF = [
    ("2025-04-30", "2025-06-01", "2025-06-30"),
    ("2025-05-31", "2025-07-01", "2025-07-31"),
    ("2025-06-30", "2025-08-01", "2025-08-31"),
    ("2025-07-31", "2025-09-01", "2025-09-30"),
    ("2025-08-31", "2025-10-01", "2025-10-31"),
    ("2025-09-30", "2025-11-01", "2025-11-30"),
]

area_snaive_results = []
area_folds_out      = []

for i, (tr_end, te_s, te_e) in enumerate(AREA_FOLDS_DEF):
    tr_end_ts = pd.Timestamp(tr_end) + pd.Timedelta("23h30m")
    te_s_ts   = pd.Timestamp(te_s)
    te_e_ts   = pd.Timestamp(te_e) + pd.Timedelta("23h30m")

    df_area_train = area_series[area_series["freeze_date"] <= tr_end_ts].copy()
    df_area_test  = area_series[
        (area_series["freeze_date"] >= te_s_ts) &
        (area_series["freeze_date"] <= te_e_ts)
    ].dropna().copy()

    area_folds_out.append({"fold": i+1, "train": df_area_train, "test": df_area_test})

    y_true = expm1_safe(df_area_test["area_kwh_log"].values)
    y_sn   = expm1_safe(df_area_test["lag_336"].values)

    m = compute_metrics(y_true, y_sn, label="Area S-Naïve")
    m["Fold"]       = i + 1
    m["Test month"] = df_area_test["freeze_date"].dt.month_name().iloc[0]
    area_snaive_results.append(m)

area_snaive_df = pd.DataFrame(area_snaive_results)
print("\n  Area S-Naïve per-fold:")
print(area_snaive_df[["Fold","Test month","MAE","RMSE","MAPE%","sMAPE%",'WAPE%']].to_string(index=False))
print(f"\n  Avg MAE  : {area_snaive_df['MAE'].mean():.5f}")
print(f"  Avg RMSE : {area_snaive_df['RMSE'].mean():.5f}")
print(f"  Avg WAPE : {area_snaive_df['WAPE%'].mean():.2f}%")

In [ ]:
# =============================================================================
# SECTION 7.2 — Area XGBoost hyperparameters
# =============================================================================

XGB_PARAMS_AREA = {
    "objective":        "reg:squarederror",
    "tree_method":      "hist",
    "n_estimators":     1000,
    "learning_rate":    0.03,
    "max_depth":        6,
    "min_child_weight": 3,
    "subsample":        0.85,
    "colsample_bytree": 0.85,
    "reg_alpha":        0.05,
    "reg_lambda":       1.0,
    "gamma":            0.01,
    "random_state":     42,
    "n_jobs":           -1,
    "early_stopping_rounds": 40,
}

print("Area-level XGBoost Hyperparameters:")
for k, v in XGB_PARAMS_AREA.items():
    print(f"  {k:<25}: {v}")

In [ ]:
# =============================================================================
# SECTION 7.3 — Area XGBoost: 6-Fold Walk-Forward Training
# Note: validation = month before test (last 30d of training window)
# =============================================================================

print("=" * 65)
print("SECTION 7.3 — Area XGBoost: 6-Fold Walk-Forward Training")
print("=" * 65)

area_fold_results = []
area_fold_models  = []
area_fold_preds   = []  # (fold, df_test with y_true/y_pred)

for fd_a in area_folds_out:
    fold_num = fd_a["fold"]
    t0 = time.time()
    print(f"\n  ── Fold {fold_num} ─────────────────────────────────────────")

    df_tr = fd_a["train"]
    df_te = fd_a["test"]

    # Use last 30 days of training window as internal validation
    val_cutoff = df_tr["freeze_date"].max() - pd.Timedelta("30d")
    df_va = df_tr[df_tr["freeze_date"] >= val_cutoff].dropna(subset=AREA_FEATURE_COLS)
    df_tr_fit = df_tr[df_tr["freeze_date"] < val_cutoff].dropna(subset=AREA_FEATURE_COLS)

    X_tr = df_tr_fit[AREA_FEATURE_COLS].values.astype("float32")
    y_tr = df_tr_fit["area_kwh_log"].values.astype("float32")
    X_va = df_va[AREA_FEATURE_COLS].values.astype("float32")
    y_va = df_va["area_kwh_log"].values.astype("float32")
    X_te = df_te[AREA_FEATURE_COLS].values.astype("float32")
    y_te = df_te["area_kwh_log"].values.astype("float32")

    model_a = xgb.XGBRegressor(**XGB_PARAMS_AREA)
    model_a.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )

    y_pred_log = model_a.predict(X_te)
    y_true_kwh = expm1_safe(y_te)
    y_pred_kwh = expm1_safe(y_pred_log)
    y_naive_kwh = None
    if "lag_336" in df_te.columns:
        y_naive_kwh = expm1_safe(df_te["lag_336"].values)

    m = compute_metrics(y_true_kwh, y_pred_kwh, label="XGBoost-Area", y_naive=y_naive_kwh)
    m["Fold"]       = fold_num
    m["Test month"] = df_te["freeze_date"].dt.month_name().iloc[0]
    m["Best iter"]  = model_a.best_iteration
    area_fold_results.append(m)

    df_te_out = df_te.copy()
    df_te_out["y_true_kwh"] = y_true_kwh
    df_te_out["y_pred_kwh"] = y_pred_kwh
    area_fold_preds.append((fold_num, df_te_out))
    area_fold_models.append(model_a)

    print(f"    Train:{len(X_tr):,}  Val:{len(X_va):,}  Test:{len(X_te):,}")
    print(f"    Best iter: {model_a.best_iteration}")
    print(f"    MAE={m['MAE']:.5f}  RMSE={m['RMSE']:.5f}  "
          f"MAPE={m['MAPE%']:.2f}%  sMAPE={m['sMAPE%']:.2f}%  "
          f"WAPE={m['WAPE%']:.2f}%  MASE={m['MASE']:.5f}  "
          f"({time.time()-t0:.1f}s)")

area_results_df = pd.DataFrame(area_fold_results)
print("\n" + "=" * 65)
print("  Area XGBoost — Walk-Forward Summary")
print("=" * 65)
print(area_results_df[["Fold","Test month","MAE","RMSE","MAPE%","sMAPE%","WAPE%","MASE","Best iter"]].to_string(index=False))
print(f"\n  Mean MAE    : {area_results_df['MAE'].mean():.5f}")
print(f"  Mean RMSE   : {area_results_df['RMSE'].mean():.5f}")
print(f"  Mean MAPE   : {area_results_df['MAPE%'].mean():.2f}%")
print(f"  Mean sMAPE  : {area_results_df['sMAPE%'].mean():.2f}%")
print(f"  Mean WAPE   : {area_results_df['WAPE%'].mean():.2f}%")
print(f"  Mean MASE   : {area_results_df['MASE'].mean():.5f}")

In [ ]:
# =============================================================================
# SECTION 7.4 — Area Model Visualisations
# =============================================================================

print("=" * 65)
print("SECTION 7.4 — Area Model Visualisations")
print("=" * 65)

# ── Fig 1: Actual vs Predicted for each fold ──────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle("Area XGBoost — Actual vs Predicted per Fold", fontweight="bold")
axes = axes.flatten()

for i, (fold_num, df_p) in enumerate(area_fold_preds):
    ax = axes[i]
    ax.plot(df_p["freeze_date"], df_p["y_true_kwh"], lw=0.8, color=C_BLUE, label="Actual")
    ax.plot(df_p["freeze_date"], df_p["y_pred_kwh"], lw=0.8, color=C_RED,
            ls="--", alpha=0.85, label="XGBoost")
    # S-Naïve overlay
    y_sn = expm1_safe(df_p["lag_336"].values)
    ax.plot(df_p["freeze_date"], y_sn, lw=0.6, color=C_GRAY,
            ls=":", alpha=0.7, label="S-Naïve")
    m = area_fold_results[i]
    ax.set_title(f"Fold {fold_num} — {m['Test month']}\nMAE={m['MAE']:.5f}  RMSE={m['RMSE']:.5f}")
    ax.set_ylabel("Area Mean kWh")
    ax.legend(fontsize=7)
    _fmt_xaxis(ax)

plt.tight_layout(); plt.show()

# ── Fig 2: Learning curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle("Area XGBoost — Validation RMSE Learning Curves", fontweight="bold")
axes = axes.flatten()

for i, (model_a, m) in enumerate(zip(area_fold_models, area_fold_results)):
    evals  = model_a.evals_result()
    val_rmse = evals["validation_0"]["rmse"]
    ax = axes[i]
    ax.plot(val_rmse, lw=1.2, color=C_BLUE)
    ax.axvline(model_a.best_iteration, color=C_RED, ls="--", lw=1,
               label=f"Best @ {model_a.best_iteration}")
    ax.set_title(f"Fold {m['Fold']} — {m['Test month']}")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Val RMSE")
    ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

# ── Fig 3: Feature Importance (last fold model) ───────────────────────────────
best_area_model = area_fold_models[-1]
fi_area = pd.DataFrame({
    "feature":    AREA_FEATURE_COLS,
    "importance": best_area_model.feature_importances_
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_fi = [C_RED if v > fi_area["importance"].quantile(0.75) else
             C_AMBER if v > fi_area["importance"].median() else C_BLUE
             for v in fi_area["importance"]]
ax.barh(fi_area["feature"], fi_area["importance"], color=colors_fi, edgecolor="white", lw=0.3)
ax.set_title("Area XGBoost — Feature Importance (Fold 6)", fontweight="bold")
ax.set_xlabel("Importance (Gain)")
plt.tight_layout(); plt.show()

# ── Fig 4: Residual analysis (last fold) ─────────────────────────────────────
last_fold_area_num, last_area_preds = area_fold_preds[-1]
resid_a = last_area_preds["y_true_kwh"].values - last_area_preds["y_pred_kwh"].values

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"Area XGBoost — Residual Analysis (Fold {last_fold_area_num})", fontweight="bold")

axes[0].hist(resid_a, bins=60, color=C_TEAL, edgecolor="white", lw=0.3)
axes[0].axvline(0, color=C_RED, lw=1.5, ls="--")
axes[0].set_title("Residual Distribution")
axes[0].set_xlabel("Residual (kWh)")

axes[1].fill_between(last_area_preds["freeze_date"], resid_a, alpha=0.5, color=C_AMBER)
axes[1].axhline(0, color=C_GRAY, lw=1)
axes[1].set_title("Residual over Time")
axes[1].set_ylabel("Residual (kWh)")
_fmt_xaxis(axes[1])

axes[2].scatter(last_area_preds["y_pred_kwh"], resid_a, s=3, alpha=0.4, color=C_BLUE)
axes[2].axhline(0, color=C_RED, lw=1, ls="--")
axes[2].set_title("Residual vs Fitted")
axes[2].set_xlabel("Fitted (kWh)")
axes[2].set_ylabel("Residual (kWh)")

plt.tight_layout(); plt.show()

## Section 8 — Area XGBoost vs S-Naïve: Comparison

In [ ]:
# =============================================================================
# SECTION 8 — Area XGBoost vs S-Naïve: Full Comparison
# =============================================================================

print("=" * 65)
print("SECTION 8 — Area XGBoost vs S-Naïve: Final Comparison")
print("=" * 65)

area_comp_df = (
    area_snaive_df[["Fold","Test month","MAE","RMSE","MAPE%","sMAPE%","WAPE%","MASE"]]
    .rename(columns={"MAE":"SN_MAE","RMSE":"SN_RMSE","MAPE%":"SN_MAPE","sMAPE%":"SN_sMAPE","WAPE%":"SN_WAPE","MASE":"SN_MASE"})
    .merge(
        area_results_df[["Fold","MAE","RMSE","MAPE%","sMAPE%","WAPE%","MASE"]]
        .rename(columns={"MAE":"XGB_MAE","RMSE":"XGB_RMSE","MAPE%":"XGB_MAPE","sMAPE%":"XGB_sMAPE","WAPE%":"XGB_WAPE","MASE":"XGB_MASE"}),
        on="Fold"
    )
)
area_comp_df["RMSE_imp%"] = (area_comp_df["SN_RMSE"] - area_comp_df["XGB_RMSE"]) / area_comp_df["SN_RMSE"] * 100
area_comp_df["MAE_imp%"]  = (area_comp_df["SN_MAE"]  - area_comp_df["XGB_MAE"])  / area_comp_df["SN_MAE"]  * 100
area_comp_df["WAPE_imp%"] = (area_comp_df["SN_WAPE"]  - area_comp_df["XGB_WAPE"])  / area_comp_df["SN_WAPE"]  * 100
area_comp_df["MASE_imp%"] = (area_comp_df["SN_MASE"]  - area_comp_df["XGB_MASE"])  / area_comp_df["SN_MASE"]  * 100

print("  Area Series: XGBoost vs S-Naïve per fold:")
print(area_comp_df.to_string(index=False))
print(f"\n  Avg RMSE improvement: {area_comp_df['RMSE_imp%'].mean():.1f}%")
print(f"  Avg MAE  improvement: {area_comp_df['MAE_imp%'].mean():.1f}%")
print(f"  Avg WAPE improvement: {area_comp_df['WAPE_imp%'].mean():.1f}%")
print(f"  Avg MASE improvement: {area_comp_df['MASE_imp%'].mean():.1f}%")

# ── Visualisation ─────────────────────────────────────────────────────────────
x        = np.arange(len(area_comp_df))
w        = 0.35
months_a = area_comp_df["Test month"].values

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle("Area XGBoost vs Seasonal Naïve — All Folds", fontweight="bold")

for ax, (sn_col, xgb_col, ylabel) in zip(
    axes.flatten(),
    [("SN_MAE","XGB_MAE","MAE (kWh)"),
     ("SN_RMSE","XGB_RMSE","RMSE (kWh)"),
     ("SN_MAPE","XGB_MAPE","MAPE (%)"),
     ("SN_sMAPE","XGB_sMAPE","sMAPE (%)"),
     ("SN_WAPE","XGB_WAPE","WAPE (%)"),
     ("SN_MASE","XGB_MASE","MASE")]
):
    ax.bar(x - w/2, area_comp_df[sn_col],  width=w, color=C_GRAY,  label="S-Naïve",  edgecolor="white")
    ax.bar(x + w/2, area_comp_df[xgb_col], width=w, color=C_TEAL,  label="XGBoost Area", edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels([f"F{f}\n{m[:3]}" for f, m in zip(area_comp_df['Fold'], months_a)], fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(f"{ylabel}")
    ax.legend()

plt.tight_layout(); plt.show()

## Section 9 — Grand Summary: Both Models vs Baseline

In [ ]:
# =============================================================================
# SECTION 9 — Final Grand Summary Dashboard
# =============================================================================

print("=" * 65)
print("SECTION 9 — Grand Summary: All Models")
print("=" * 65)

# ── Summary table ─────────────────────────────────────────────────────────────
def avg_row(df, label):
    return {
        "Model": label,
        "MAE":    df["MAE"].mean(),
        "RMSE":   df["RMSE"].mean(),
        "MAPE%":  df["MAPE%"].mean(),
        "sMAPE%": df["sMAPE%"].mean(),
        "WAPE%": df["WAPE%"].mean(),
        "MASE":   df["MASE"].mean() if "MASE" in df.columns else np.nan,
    }

summary = pd.DataFrame([
    avg_row(snaive_df,       "Per-Meter S-Naïve"),
    avg_row(meter_results_df,"Per-Meter XGBoost"),
    avg_row(area_snaive_df,  "Area S-Naïve"),
    avg_row(area_results_df, "Area XGBoost"),
])

print("\n  Average metrics across all 6 folds:")
print(summary.to_string(index=False))

# ── Visual dashboard ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Grand Summary — All Models Averaged over 6 Walk-Forward Folds",
             fontweight="bold", fontsize=13)

model_colors = [C_GRAY, C_BLUE, C_AMBER, C_TEAL]

for ax, metric in zip(axes, ["MAE","RMSE","MAPE%","sMAPE%","WAPE%","MASE"]):
    bars = ax.bar(summary["Model"], summary[metric],
                  color=model_colors, edgecolor="white", lw=0.3)
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, v * 1.01,
                f"{v:.4f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(metric)
    ax.set_xticklabels(summary["Model"], rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(metric)

plt.tight_layout(); plt.show()

# ── Per-fold RMSE trajectory ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("RMSE Trajectory across Walk-Forward Folds", fontweight="bold")

fold_nums = [1, 2, 3, 4, 5, 6]
ax.plot(fold_nums, snaive_df["RMSE"].values,         marker="o", lw=1.5, color=C_GRAY,  label="Per-Meter S-Naïve")
ax.plot(fold_nums, meter_results_df["RMSE"].values,   marker="s", lw=1.5, color=C_BLUE,  label="Per-Meter XGBoost")
ax.plot(fold_nums, area_snaive_df["RMSE"].values,     marker="^", lw=1.5, color=C_AMBER, label="Area S-Naïve",  ls="--")
ax.plot(fold_nums, area_results_df["RMSE"].values,    marker="D", lw=1.5, color=C_TEAL,  label="Area XGBoost",  ls="--")
ax.set_xlabel("Fold")
ax.set_ylabel("RMSE (kWh)")
ax.set_xticks(fold_nums)
ax.set_xticklabels([f"F{f}\n{m[:3]}" for f, m in
                    zip(fold_nums, meter_results_df["Test month"].values)], fontsize=8)
ax.legend()
plt.tight_layout(); plt.show()

# ── Final bill estimation stats ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("  Monthly Bill Estimation — Final Fold Summary")
print("=" * 65)
print(f"  Test month            : {folds_out[-1]['test']['freeze_date'].dt.month_name().iloc[0]} 2025")
print(f"  Meters evaluated      : {len(monthly_pred)}")
print(f"  Tier accuracy         : {tier_accuracy:.1f}%")
print(f"  Mean |bill error|     : {mean_bill_err:.3f} JOD")
print(f"  Median |bill error|   : {median_bill_err:.3f} JOD")
print()
print("  Tier breakdown:")
for tier in tier_order:
    n_actual   = (monthly_pred["actual_tier"]   == tier).sum()
    n_forecast = (monthly_pred["forecast_tier"] == tier).sum()
    print(f"    {tier:<25}: Actual={n_actual:>4}  Forecast={n_forecast:>4}")

## Section 10 — Next-Month Inference Helper

A ready-to-use function that forecasts the next month for any meter using the trained model.

In [ ]:
# =============================================================================
# SECTION 10 — Production-Ready Inference Helper
# =============================================================================

def forecast_meter_next_month(
    meter_id: str,
    df_history: pd.DataFrame,
    model: xgb.XGBRegressor,
    feature_cols: list,
    meter_encoder: dict,
) -> dict:
    """
    Given a meter's history (preprocessed, log1p-scaled, feature-engineered),
    predict the NEXT month's consumption, estimate the bill, and assign the tier.

    Parameters
    ----------
    meter_id      : meter identifier (e.g. 'M0042')
    df_history    : the full df_scaled filtered to this meter
    model         : trained XGBRegressor (from meter_fold_models[-1])
    feature_cols  : FEATURES_PER_METER
    meter_encoder : meter_enc dict

    Returns
    -------
    dict with forecast_kwh_monthly, bill_jod, tier, and a per-slot DataFrame
    """
    sub = df_history[df_history["meter_B"] == meter_id].sort_values("freeze_date").copy()
    if sub.empty:
        raise ValueError(f"Meter {meter_id} not found in history.")

    sub["meter_id"] = meter_encoder.get(meter_id, 0)
    feat_with_id    = feature_cols + ["meter_id"]
    sub_clean       = sub.dropna(subset=feat_with_id)

    X   = sub_clean[feat_with_id].values.astype("float32")
    y_log_pred = model.predict(X)
    y_kwh_pred = expm1_safe(y_log_pred)
    y_kwh_true = expm1_safe(sub_clean[TARGET].values)

    # Monthly aggregation
    monthly_kwh_pred  = float(y_kwh_pred.sum())
    monthly_kwh_true  = float(y_kwh_true.sum())
    bill_jod_pred     = calc_bill_jod(monthly_kwh_pred)
    bill_jod_true     = calc_bill_jod(monthly_kwh_true)
    tier_pred         = tier_label(monthly_kwh_pred)
    tier_true         = tier_label(monthly_kwh_true)

    # Per-slot results
    slot_df = sub_clean[["freeze_date",TARGET]].copy()
    slot_df["actual_kwh"]    = y_kwh_true
    slot_df["forecast_kwh"]  = y_kwh_pred

    return {
        "meter_id":            meter_id,
        "n_slots_forecast":    len(y_kwh_pred),
        "forecast_kwh_monthly":monthly_kwh_pred,
        "actual_kwh_monthly":  monthly_kwh_true,
        "forecast_bill_jod":   bill_jod_pred,
        "actual_bill_jod":     bill_jod_true,
        "forecast_tier":       tier_pred,
        "actual_tier":         tier_true,
        "slot_df":             slot_df,
    }


# ── Demo: run on 3 random meters from the last test fold ─────────────────────
demo_model = meter_fold_models[-1]
last_fold_num_m, last_preds_m = meter_fold_preds[-1]
demo_meters = last_preds_m["meter_B"].unique()[:3]

print("=" * 65)
print("SECTION 10 — Per-Meter Inference Demo (Last Test Fold)")
print("=" * 65)

fig, axes = plt.subplots(len(demo_meters), 1, figsize=(14, 5 * len(demo_meters)))
fig.suptitle("Per-Meter Forecast Demo — Next Month Predicted vs Actual", fontweight="bold")
if len(demo_meters) == 1: axes = [axes]

for ax, mid in zip(axes, demo_meters):
    result = forecast_meter_next_month(
        mid,
        folds_out[-1]["test"],   # last test split as 'history'
        demo_model,
        FEATURES_PER_METER,
        meter_enc,
    )
    sd = result["slot_df"]

    ax.plot(sd["freeze_date"], sd["actual_kwh"],   lw=0.8, color=C_BLUE,  label="Actual kWh")
    ax.plot(sd["freeze_date"], sd["forecast_kwh"], lw=0.8, color=C_RED, ls="--", label="Forecast kWh")

    title = (f"Meter {mid} │ "
             f"Forecast: {result['forecast_kwh_monthly']:.1f} kWh  "
             f"({result['forecast_bill_jod']:.2f} JOD, {result['forecast_tier']}) │ "
             f"Actual: {result['actual_kwh_monthly']:.1f} kWh  "
             f"({result['actual_bill_jod']:.2f} JOD, {result['actual_tier']})")
    ax.set_title(title, fontsize=9)
    ax.set_ylabel("kWh / 30-min"); ax.legend(); _fmt_xaxis(ax)

    print(f"\n  Meter: {result['meter_id']}")
    print(f"    Forecast kWh : {result['forecast_kwh_monthly']:.2f}   "
          f"Actual kWh : {result['actual_kwh_monthly']:.2f}")
    print(f"    Forecast bill: {result['forecast_bill_jod']:.3f} JOD   "
          f"Actual bill: {result['actual_bill_jod']:.3f} JOD")
    print(f"    Forecast tier: {result['forecast_tier']}   "
          f"Actual tier: {result['actual_tier']}")

plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# SECTION 11 — Save Models & Results
# =============================================================================

import json

os.makedirs("models", exist_ok=True)

# Save best (last fold) XGBoost models in native format
meter_fold_models[-1].save_model("models/xgb_per_meter_fold6.ubj")
area_fold_models[-1].save_model("models/xgb_area_fold6.ubj")

# Save evaluation results
meter_results_df.to_csv("models/per_meter_xgb_results.csv", index=False)
area_results_df.to_csv("models/area_xgb_results.csv",       index=False)
snaive_df.to_csv("models/snaive_per_meter_results.csv",     index=False)
area_snaive_df.to_csv("models/snaive_area_results.csv",     index=False)
monthly_pred.to_csv("models/monthly_forecast_last_fold.csv", index=False)

# Save meter encoder
with open("models/meter_encoder.json", "w") as f:
    json.dump(meter_enc, f)

print("✔ Models and results saved to ./models/")
print("  xgb_per_meter_fold6.ubj   — per-meter XGBoost (last fold)")
print("  xgb_area_fold6.ubj        — area XGBoost (last fold)")
print("  per_meter_xgb_results.csv — walk-forward metrics")
print("  area_xgb_results.csv      — walk-forward metrics (area)")
print("  monthly_forecast_last_fold.csv — bill + tier forecasts")

In [ ]:
os.makedirs("models", exist_ok=True)

with open("models/feature_cols.json", "w") as f:
    json.dump(FEATURES_PER_METER, f, indent=2)

print("Saved feature columns to models/feature_cols.json")